# Projeto Integrador - Analise de Vendas do Varejo Brasileiro

**Curso:** Ciencia de Dados e Inteligencia Artificial
**Disciplina:** Linguagem de Programacao 6

---

### Objetivo

Este projeto integra **todas as 12 aulas** da disciplina em uma analise completa e continua, desde a importacao dos dados brutos ate a publicacao de um dashboard interativo.

### Pipeline de Dados

```
Problema -> Coleta -> Limpeza -> Analise -> Visualizacao -> Dashboard -> Publicacao
```

### Fluxo do Projeto

```
vendas_brasil.csv (dados brutos)
    | Secao 3 - Limpeza
vendas_brasil_clean.csv (dados tratados)
    | Secao 7 - SQLAlchemy
vendas.db (banco SQLite)
    | Secoes 8-9 - Streamlit
app_dashboard.py (aplicacao web)
```

---

## Secao 1: Introducao e Setup - Do Dado Bruto a Decisao Estrategica

*Corresponde a Aula 1: Python para Analise de Dados*

### O que vamos fazer nesta secao:
1. Importar todas as bibliotecas necessarias
2. Carregar o dataset `vendas_brasil.csv`
3. Primeira visao geral dos dados

### Conceito-chave: Pipeline de Dados

Todo projeto de analise segue um fluxo:
- **Problema** -> O que queremos responder?
- **Coleta** -> De onde vem os dados?
- **Limpeza** -> Os dados estao prontos para analise?
- **Analise** -> Quais padroes existem?
- **Visualizacao** -> Como comunicar os resultados?
- **Dashboard** -> Como tornar acessivel?
- **Publicacao** -> Como chegar ao usuario final?

In [ ]:
# ============================================================
# SECAO 1 - Importacao das Bibliotecas
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Bibliotecas importadas com sucesso!')

In [ ]:
# ============================================================
# SECAO 1 - Carregamento dos Dados
# ============================================================

df = pd.read_csv('vendas_brasil.csv')
print(f'Dataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'Colunas: {list(df.columns)}')
df.head(10)

In [ ]:
# Visao geral rapida
print('=' * 50)
print('VISAO GERAL DO DATASET')
print('=' * 50)
print(f'Formato: {df.shape}')
print(f'Colunas e tipos:')
print(df.dtypes)
print(f'Valores nulos por coluna:')
print(df.isnull().sum())
print(f'Linhas duplicadas: {df.duplicated().sum()}')

---

## Secao 2: Exploracao Inicial - Dominando o Basico que Resolve

*Corresponde a Aula 2: Pandas Essencial*

### O que vamos fazer nesta secao:
1. Check-up da saude dos dados (shape, info, dtypes)
2. Selecao de colunas e filtragem por regras de negocio
3. Ordenacao para priorizacao gerencial (Top N)

### Conceito-chave: Boas Praticas de Exploracao
- Sempre comece pelo `shape` para entender o tamanho
- Use `info()` para verificar tipos e nulos
- `describe()` para estatisticas descritivas rapidas
- Filtre antes de agrupar para respostas especificas

In [ ]:
# ============================================================
# SECAO 2 - Check-up Inicial da Saude dos Dados
# ============================================================

print('=' * 50)
print('CHECK-UP DA SAUDE DOS DADOS')
print('=' * 50)

print(f'Registros: {df.shape[0]:,}')
print(f'Colunas: {df.shape[1]}')
print(f'Tipos de dados:')
for col in df.columns:
    tipo = df[col].dtype
    nulos = df[col].isnull().sum()
    unicos = df[col].nunique()
    print(f'  {col:20s} | {str(tipo):10s} | {nulos:4d} nulos | {unicos:5d} unicos')

In [ ]:
# ============================================================
# SECAO 2 - Estatisticas Descritivas
# ============================================================

df.describe()

In [ ]:
# ============================================================
# SECAO 2 - Selecao de Colunas
# ============================================================

# Selecionar apenas colunas numericas
colunas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Colunas numericas: {colunas_numericas}')

# Selecionar apenas colunas categoricas
colunas_cat = df.select_dtypes(include=['object']).columns.tolist()
print(f'Colunas categoricas: {colunas_cat}')

# Selecionar colunas especificas
df_preview = df[['data', 'uf', 'canal', 'categoria', 'receita', 'lucro']]
df_preview.head()

In [ ]:
# ============================================================
# SECAO 2 - Filtragem por Regras de Negocio
# ============================================================

# Vendas do estado de Sao Paulo
df_sp = df[df['uf'] == 'SP']
print(f'Vendas em SP: {len(df_sp):,} registros')

# Vendas online com lucro positivo
df_lucro = df[(df['canal'] == 'Online') & (df['lucro'] > 0)]
print(f'Vendas Online com lucro positivo: {len(df_lucro):,} registros')

# Receita acima de R$ 1.000
df_altovalor = df[df['receita'] > 1000]
print(f'Vendas com receita > R$ 1.000: {len(df_altovalor):,} registros')

In [ ]:
# ============================================================
# SECAO 2 - Ordenacao para Priorizacao Gerencial
# ============================================================

# Top 10 maiores vendas por receita
print('TOP 10 MAIORES VENDAS POR RECEITA')
print('-' * 50)
top10 = df.nlargest(10, 'receita')[['data', 'uf', 'canal', 'categoria', 'produto', 'receita']]
top10

In [ ]:
# Valores unicos das colunas categoricas
print('VALORES UNICOS DAS COLUNAS CATEGORICAS')
print('=' * 50)
for col in colunas_cat:
    if col != 'data':
        print(f'  {col.upper()}:')
        print(f'   {df[col].unique()}')

---

## Secao 3: Limpeza e Preparacao - A Fundacao Invisivel

*Corresponde a Aula 3: Limpeza, Preparacao e Qualidade dos Dados*

### O que vamos fazer nesta secao:
1. Conversao e correcao de tipos de dados
2. Estrategias para lidar com valores ausentes
3. Engenharia de recursos (feature engineering)
4. Exportacao da base tratada

### Conceito-chave: Regra dos 80%

**80% do tempo** de um cientista de dados e gasto em limpeza e preparacao. Dados limpos sao a base de qualquer analise confiavel.

In [ ]:
# ============================================================
# SECAO 3 - Conversao de Tipos de Dados
# ============================================================

# Converter data para datetime
df['data'] = pd.to_datetime(df['data'])
print(f'Tipo da coluna data: {df["data"].dtype}')
print(f'Periodo: {df["data"].min().strftime("%Y-%m-%d")} a {df["data"].max().strftime("%Y-%m-%d")}')

# Garantir tipos numericos corretos
colunas_valores = ['quantidade', 'receita', 'custo', 'lucro']
for col in colunas_valores:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f'  {col}: {df[col].dtype}')

In [ ]:
# ============================================================
# SECAO 3 - Tratamento de Valores Ausentes
# ============================================================

print('ANTES do tratamento:')
print(f'  Total de nulos: {df.isnull().sum().sum()}')
print(f'  Por coluna:')
for col in df.columns:
    n = df[col].isnull().sum()
    if n > 0:
        print(f'    {col}: {n} nulos ({n/len(df)*100:.1f}%)')

# Preencher valores numericos com a mediana (robusta a outliers)
for col in colunas_valores:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)

print(f'\nDEPOIS do tratamento:')
print(f'  Total de nulos: {df.isnull().sum().sum()}')

In [ ]:
# ============================================================
# SECAO 3 - Engenharia de Recursos (Feature Engineering)
# ============================================================

# Extrair componentes da data
df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month
df['mes_nome'] = df['data'].dt.month_name()
df['trimestre'] = df['data'].dt.quarter
df['dia_semana'] = df['data'].dt.day_name()

# Calcular margem de lucro
df['margem_lucro'] = ((df['receita'] - df['custo']) / df['receita'] * 100).round(2)

# Calcular ticket unitario
df['ticket_unitario'] = (df['receita'] / df['quantidade']).round(2)

# Classificar lucro
df['status_lucro'] = df['lucro'].apply(lambda x: 'Lucrativo' if x > 0 else 'Prejuizo')

print('Novas colunas criadas: ano, mes, mes_nome, trimestre, dia_semana, margem_lucro, ticket_unitario, status_lucro')
df[['data', 'ano', 'mes', 'mes_nome', 'trimestre', 'margem_lucro', 'ticket_unitario', 'status_lucro']].head()

In [ ]:
# ============================================================
# SECAO 3 - Remocao de Inconsistencias
# ============================================================

# Verificar registros com valores negativos suspeitos
negativos = df[(df['quantidade'] < 0) | (df['receita'] < 0)]
print(f'Registros com valores negativos: {len(negativos)}')

# Remover registros onde quantidade ou receita sao zero/negativo
antes = len(df)
df = df[(df['quantidade'] > 0) & (df['receita'] > 0)]
depois = len(df)
print(f'Registros removidos: {antes - depois}')
print(f'Registros restantes: {len(df)}')

In [ ]:
# ============================================================
# SECAO 3 - Exportacao da Base Tratada
# ============================================================

df.to_csv('vendas_brasil_clean.csv', index=False, encoding='utf-8')
print(f'Arquivo exportado: vendas_brasil_clean.csv')
print(f'  Registros: {len(df):,}')
print(f'  Colunas: {len(df.columns)}')
print(f'  Tamanho: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

---

## Secao 4: KPIs e Indicadores de Negocio

*Corresponde a Aula 4: Analise de Dados e Construcao de Indicadores*

### O que vamos fazer nesta secao:
1. Calcular os 4 KPIs fundamentais do varejo
2. Fatiamento de dados por dimensoes
3. Agregacoes com groupby
4. Rankings de produtos campeoes e viloes

### Conceito-chave: KPIs do Varejo

| KPI | Formula | O que mede |
|-----|---------|------------|
| Receita Total | SUM(receita) | Volume de vendas |
| Lucro Total | SUM(lucro) | Rentabilidade |
| Margem de Lucro % | (Lucro/Receita)*100 | Eficiencia |
| Ticket Medio | Receita / Qtd Vendas | Valor medio por transacao |

In [ ]:
# ============================================================
# SECAO 4 - Calculo dos 4 KPIs Fundamentais
# ============================================================

print('=' * 60)
print('KPIs FUNDAMENTAIS DO VAREJO')
print('=' * 60)

receita_total = df['receita'].sum()
lucro_total = df['lucro'].sum()
margem_media = (lucro_total / receita_total * 100)
ticket_medio = df['receita'].mean()
qtd_vendas = len(df)

print(f'Receita Total:     R$ {receita_total:>15,.2f}')
print(f'Lucro Total:       R$ {lucro_total:>15,.2f}')
print(f'Margem de Lucro:   {margem_media:>14.1f}%')
print(f'Ticket Medio:      R$ {ticket_medio:>15,.2f}')
print(f'Total de Vendas:   {qtd_vendas:>15,}')
print('=' * 60)

In [ ]:
# ============================================================
# SECAO 4 - KPIs por Canal de Venda
# ============================================================

kpi_canal = df.groupby('canal').agg(
    receita=('receita', 'sum'),
    lucro=('lucro', 'sum'),
    qtd_vendas=('receita', 'count'),
    ticket_medio=('receita', 'mean')
).round(2)

kpi_canal['margem_%'] = (kpi_canal['lucro'] / kpi_canal['receita'] * 100).round(1)
kpi_canal = kpi_canal.sort_values('receita', ascending=False)
print('KPIs POR CANAL DE VENDA')
print('-' * 60)
kpi_canal

In [ ]:
# ============================================================
# SECAO 4 - KPIs por Categoria
# ============================================================

kpi_cat = df.groupby('categoria').agg(
    receita=('receita', 'sum'),
    lucro=('lucro', 'sum'),
    qtd=('receita', 'count')
).round(2)

kpi_cat['margem_%'] = (kpi_cat['lucro'] / kpi_cat['receita'] * 100).round(1)
kpi_cat = kpi_cat.sort_values('receita', ascending=False)
print('KPIs POR CATEGORIA')
print('-' * 60)
kpi_cat

In [ ]:
# ============================================================
# SECAO 4 - KPIs por Estado (UF)
# ============================================================

kpi_uf = df.groupby('uf').agg(
    receita=('receita', 'sum'),
    lucro=('lucro', 'sum'),
    qtd=('receita', 'count')
).round(2)

kpi_uf['margem_%'] = (kpi_uf['lucro'] / kpi_uf['receita'] * 100).round(1)
kpi_uf['ticket_medio'] = (kpi_uf['receita'] / kpi_uf['qtd']).round(2)
kpi_uf = kpi_uf.sort_values('receita', ascending=False)
print('KPIs POR ESTADO')
print('-' * 60)
kpi_uf

In [ ]:
# ============================================================
# SECAO 4 - Evolucao Mensal dos KPIs
# ============================================================

df['ano_mes'] = df['data'].dt.to_period('M')

kpi_mensal = df.groupby('ano_mes').agg(
    receita=('receita', 'sum'),
    lucro=('lucro', 'sum'),
    qtd=('receita', 'count')
).round(2)

kpi_mensal['margem_%'] = (kpi_mensal['lucro'] / kpi_mensal['receita'] * 100).round(1)
kpi_mensal['ticket_medio'] = (kpi_mensal['receita'] / kpi_mensal['qtd']).round(2)

print('EVOLUCAO MENSAL DOS KPIs')
print('-' * 70)
kpi_mensal

In [ ]:
# ============================================================
# SECAO 4 - Ranking: Produtos Campeoes e Viloes
# ============================================================

ranking_prod = df.groupby('produto').agg(
    receita=('receita', 'sum'),
    lucro=('lucro', 'sum'),
    qtd=('receita', 'count')
).round(2).sort_values('receita', ascending=False)

print('TOP 5 PRODUTOS CAMPEOES (maior receita)')
print('-' * 50)
print(ranking_prod.head(5))
print()
print('TOP 5 PRODUTOS VILAOES (menor lucro)')
print('-' * 50)
print(ranking_prod.sort_values('lucro').head(5))

---

## Secao 5: Visualizacao Estatica - Matplotlib e Seaborn

*Corresponde a Aula 5: Visualizacao de Dados para Decisoes*

### O que vamos fazer nesta secao:
1. Graficos de barras para comparacao
2. Graficos de linhas para tendencias temporais
3. Boxplot para distribuicao
4. Heatmap para correlacao
5. Scatter para relacao entre variaveis

### Conceito-chave: Escolha do Grafico

| Intencao Analitica | Grafico Adequado |
|--------------------|------------------|
| Comparacao | Barras |
| Tendencia Temporal | Linhas |
| Distribuicao | Histograma / Boxplot |
| Correlacao | Scatter |
| Composicao | Pizza (com cuidado) |

In [ ]:
# ============================================================
# SECAO 5 - Grafico de Barras: Receita por Canal
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras simples
canal_receita = df.groupby('canal')['receita'].sum().sort_values(ascending=True)
canal_receita.plot(kind='barh', ax=axes[0], color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'])
axes[0].set_title('Receita Total por Canal de Venda', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Receita (R$)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))

# Barras com margem
canal_margem = df.groupby('canal')['lucro'].sum().sort_values(ascending=True)
canal_margem.plot(kind='barh', ax=axes[1], color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'])
axes[1].set_title('Lucro Total por Canal de Venda', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Lucro (R$)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECAO 5 - Grafico de Linhas: Evolucao Temporal
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Evolucao da receita mensal
receita_mensal = df.groupby('ano_mes')['receita'].sum()
receita_mensal.plot(ax=axes[0], marker='o', linewidth=2, color='#2196F3')
axes[0].set_title('Evolucao Mensal da Receita', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Receita (R$)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
axes[0].grid(True, alpha=0.3)

# Evolucao do lucro mensal
lucro_mensal = df.groupby('ano_mes')['lucro'].sum()
lucro_mensal.plot(ax=axes[1], marker='s', linewidth=2, color='#4CAF50')
axes[1].set_title('Evolucao Mensal do Lucro', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Lucro (R$)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECAO 5 - Boxplot: Distribuicao da Receita
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot por canal
df.boxplot(column='receita', by='canal', ax=axes[0])
axes[0].set_title('Distribuicao da Receita por Canal', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Canal')
axes[0].set_ylabel('Receita (R$)')
plt.sca(axes[0])
plt.title('')

# Boxplot por categoria
df.boxplot(column='receita', by='categoria', ax=axes[1])
axes[1].set_title('Distribuicao da Receita por Categoria', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Categoria')
axes[1].set_ylabel('Receita (R$)')
plt.sca(axes[1])
plt.title('')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECAO 5 - Heatmap: Correlacao entre Variaveis
# ============================================================

cols_corr = ['quantidade', 'receita', 'custo', 'lucro', 'margem_lucro', 'ticket_unitario']
matriz_corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matriz_corr, annot=True, cmap='RdYlGn', center=0,
            fmt='.2f', linewidths=0.5, ax=ax,
            square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Mapa de Correlacao entre Variaveis Numericas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECAO 5 - Scatter: Receita vs Lucro
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter colorido por canal
cores = {'Online': '#2196F3', 'Loja Fisica': '#4CAF50', 'Marketplace': '#FF9800', 'Representante': '#9C27B0'}
for canal, cor in cores.items():
    dados = df[df['canal'] == canal]
    axes[0].scatter(dados['receita'], dados['lucro'], c=cor, label=canal, alpha=0.6, s=30)
axes[0].set_title('Receita vs Lucro por Canal', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Receita (R$)')
axes[0].set_ylabel('Lucro (R$)')
axes[0].legend()
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Scatter colorido por categoria
cat_unicas = df['categoria'].unique()
palette = plt.cm.Set3(np.linspace(0, 1, len(cat_unicas)))
for i, cat in enumerate(cat_unicas):
    dados = df[df['categoria'] == cat]
    axes[1].scatter(dados['receita'], dados['lucro'], c=[palette[i]], label=cat, alpha=0.6, s=30)
axes[1].set_title('Receita vs Lucro por Categoria', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Receita (R$)')
axes[1].set_ylabel('Lucro (R$)')
axes[1].legend(fontsize=8, loc='best')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECAO 5 - Histograma: Distribuicao da Receita
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['receita'], bins=30, color='#2196F3', edgecolor='white', alpha=0.8)
ax.axvline(df['receita'].mean(), color='red', linestyle='--', label=f'Media: R$ {df["receita"].mean():,.2f}')
ax.axvline(df['receita'].median(), color='green', linestyle='--', label=f'Mediana: R$ {df["receita"].median():,.2f}')
ax.set_title('Distribuicao da Receita', fontsize=14, fontweight='bold')
ax.set_xlabel('Receita (R$)')
ax.set_ylabel('Frequencia')
ax.legend()
plt.tight_layout()
plt.show()

---

## Secao 6: Visualizacao Interativa - Plotly

*Corresponde a Aula 6: Visualizacao Interativa com Plotly*

### O que vamos fazer nesta secao:
1. Graficos interativos com tooltips (hover)
2. Zoom e pan interativos
3. Filtros por legenda (click)
4. Series temporais interativas

### Conceito-chave: Tridada da Interatividade

- **Tooltips/Hover**: Informacoes ao passar o mouse
- **Zoom & Pan**: Navegacao livre nos dados
- **Filtros por Legenda**: Click para mostrar/esconder series

In [ ]:
# ============================================================
# SECAO 6 - Grafico de Linhas Interativo (Plotly Express)
# ============================================================

receita_mes_canal = df.groupby(['ano_mes', 'canal'])['receita'].sum().reset_index()
receita_mes_canal['ano_mes_str'] = receita_mes_canal['ano_mes'].astype(str)

fig = px.line(receita_mes_canal, x='ano_mes_str', y='receita', color='canal',
              title='Evolucao Mensal da Receita por Canal',
              labels={'ano_mes_str': 'Mes', 'receita': 'Receita (R$)', 'canal': 'Canal'},
              markers=True)
fig.update_layout(hovermode='x unified')
fig.update_traces(hovertemplate='Mes: %{x}<br>Receita: R$ %{y:,.2f}<extra></extra>')
fig.show()

In [ ]:
# ============================================================
# SECAO 6 - Grafico de Barras Agrupadas Interativo
# ============================================================

cat_canal = df.groupby(['categoria', 'canal'])['receita'].sum().reset_index()

fig = px.bar(cat_canal, x='categoria', y='receita', color='canal',
             title='Receita por Categoria e Canal',
             labels={'categoria': 'Categoria', 'receita': 'Receita (R$)', 'canal': 'Canal'},
             barmode='group')
fig.update_layout(xaxis_tickangle=-45)
fig.update_traces(hovertemplate='Categoria: %{x}<br>Receita: R$ %{y:,.2f}<br>Canal: %{legenddata}<extra></extra>')
fig.show()

In [ ]:
# ============================================================
# SECAO 6 - Scatter Interativo
# ============================================================

fig = px.scatter(df, x='receita', y='lucro', color='canal', size='quantidade',
                 hover_data=['uf', 'categoria', 'produto'],
                 title='Receita vs Lucro (tamanho = quantidade)',
                 labels={'receita': 'Receita (R$)', 'lucro': 'Lucro (R$)', 'canal': 'Canal'},
                 opacity=0.7)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.show()

In [ ]:
# ============================================================
# SECAO 6 - Treemap: Composicao Hierarquica
# ============================================================

treemap_data = df.groupby(['uf', 'categoria'])['receita'].sum().reset_index()

fig = px.treemap(treemap_data, path=['uf', 'categoria'], values='receita',
                 title='Composicao da Receita por Estado e Categoria',
                 color='receita', color_continuous_scale='Blues')
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig.show()

In [ ]:
# ============================================================
# SECAO 6 - Grafico de Caixa Interativo
# ============================================================

fig = px.box(df, x='canal', y='receita', color='canal',
             title='Distribuicao da Receita por Canal de Venda',
             labels={'canal': 'Canal', 'receita': 'Receita (R$)'})
fig.update_layout(showlegend=False)
fig.show()

---

## Secao 7: SQL + Python - A Refinaria de Dados

*Corresponde a Aula 7: Integracao SQL + Python*

### O que vamos fazer nesta secao:
1. Criar um banco SQLite com tabelas Fato e Dimensao
2. Inserir dados do DataFrame no banco
3. Executar consultas SQL diretamente pelo Python

### Conceito-chave: Modelagem Relacional

```
dim_produto (produto_id, nome, categoria)
dim_canal   (canal_id, nome_canal)
dim_uf      (uf_id, uf, regiao)
fato_vendas (venda_id, data, produto_id, canal_id, uf_id, quantidade, receita, custo, lucro)
```

In [ ]:
# ============================================================
# SECAO 7 - Criar Banco SQLite e Tabelas
# ============================================================

engine = create_engine('sqlite:///vendas.db', echo=False)

with engine.connect() as conn:
    # Tabela dimensao: Produto
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS dim_produto (
            produto_id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome_produto TEXT UNIQUE,
            categoria TEXT
        )
    '''))

    # Tabela dimensao: Canal
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS dim_canal (
            canal_id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome_canal TEXT UNIQUE
        )
    '''))

    # Tabela dimensao: UF
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS dim_uf (
            uf_id INTEGER PRIMARY KEY AUTOINCREMENT,
            uf TEXT UNIQUE,
            regiao TEXT
        )
    '''))

    # Tabela fato: Vendas
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS fato_vendas (
            venda_id INTEGER PRIMARY KEY AUTOINCREMENT,
            data TEXT,
            produto_id INTEGER,
            canal_id INTEGER,
            uf_id INTEGER,
            quantidade REAL,
            receita REAL,
            custo REAL,
            lucro REAL,
            FOREIGN KEY (produto_id) REFERENCES dim_produto(produto_id),
            FOREIGN KEY (canal_id) REFERENCES dim_canal(canal_id),
            FOREIGN KEY (uf_id) REFERENCES dim_uf(uf_id)
        )
    '''))
    conn.commit()

print('Tabelas criadas com sucesso!')

In [ ]:
# ============================================================
# SECAO 7 - Inserir Dados nas Tabelas Dimensao
# ============================================================

with engine.connect() as conn:
    # Inserir produtos
    produtos = df[['produto', 'categoria']].drop_duplicates()
    for _, row in produtos.iterrows():
        conn.execute(text('INSERT OR IGNORE INTO dim_produto (nome_produto, categoria) VALUES (:p, :c)'),
                     {'p': row['produto'], 'c': row['categoria']})

    # Inserir canais
    for canal in df['canal'].unique():
        conn.execute(text('INSERT OR IGNORE INTO dim_canal (nome_canal) VALUES (:n)'), {'n': canal})

    # Inserir UFs
    for uf in df['uf'].unique():
        conn.execute(text('INSERT OR IGNORE INTO dim_uf (uf) VALUES (:u)'), {'u': uf})

    conn.commit()
print('Dados inseridos nas tabelas dimensao!')

In [ ]:
# ============================================================
# SECAO 7 - Inserir Dados na Tabela Fato
# ============================================================

with engine.connect() as conn:
    # Buscar IDs das dimensoes
    produtos_map = {row[0]: row[1] for row in conn.execute(text('SELECT nome_produto, produto_id FROM dim_produto'))}
    canais_map = {row[0]: row[1] for row in conn.execute(text('SELECT nome_canal, canal_id FROM dim_canal'))}
    ufs_map = {row[0]: row[1] for row in conn.execute(text('SELECT uf, uf_id FROM dim_uf'))}

    # Inserir vendas
    for _, row in df.iterrows():
        conn.execute(text('''
            INSERT INTO fato_vendas (data, produto_id, canal_id, uf_id, quantidade, receita, custo, lucro)
            VALUES (:data, :pid, :cid, :uid, :qtd, :rec, :cust, :luc)
        '''), {
            'data': str(row['data'].date()),
            'pid': produtos_map.get(row['produto']),
            'cid': canais_map.get(row['canal']),
            'uid': ufs_map.get(row['uf']),
            'qtd': row['quantidade'],
            'rec': row['receita'],
            'cust': row['custo'],
            'luc': row['lucro']
        })
    conn.commit()

print(f'Tabela fato_vendas populada com {len(df)} registros!')

In [ ]:
# ============================================================
# SECAO 7 - Consultas SQL via Python
# ============================================================

with engine.connect() as conn:
    # Consulta 1: Receita total por canal
    resultado = conn.execute(text('''
        SELECT c.nome_canal, 
               COUNT(*) as qtd_vendas,
               SUM(f.receita) as receita_total,
               SUM(f.lucro) as lucro_total
        FROM fato_vendas f
        JOIN dim_canal c ON f.canal_id = c.canal_id
        GROUP BY c.nome_canal
        ORDER BY receita_total DESC
    '''))

    print('RECEITA POR CANAL (via SQL)')
    print('-' * 60)
    for row in resultado:
        print(f'  {row[0]:20s} | {row[1]:5d} vendas | R$ {row[2]:>12,.2f} | R$ {row[3]:>12,.2f}')

In [ ]:
# ============================================================
# SECAO 7 - Ingestao de Dados SQL para DataFrame
# ============================================================

# Ler resultado de uma query diretamente para DataFrame
df_sql = pd.read_sql('''
    SELECT u.uf, c.nome_canal, p.categoria,
           SUM(f.receita) as receita_total,
           SUM(f.lucro) as lucro_total,
           COUNT(*) as qtd_vendas
    FROM fato_vendas f
    JOIN dim_uf u ON f.uf_id = u.uf_id
    JOIN dim_canal c ON f.canal_id = c.canal_id
    JOIN dim_produto p ON f.produto_id = p.produto_id
    GROUP BY u.uf, c.nome_canal, p.categoria
    ORDER BY receita_total DESC
''', engine)

print('Dados ingeridos do SQLite para DataFrame!')
print(f'Shape: {df_sql.shape}')
df_sql.head(10)

---

## Secao 8-9: Dashboard Analitico com Streamlit

*Corresponde as Aulas 8 e 9: Dashboards Analiticos e Profissionais*

### O que vamos fazer nesta secao:
1. Estruturar um app Streamlit completo
2. Criar sidebar com filtros
3. Exibir KPIs com st.metric e deltas
4. Renderizar graficos Plotly
5. Storytelling analitico com markdown

### Conceito-chave: Layout Profissional

```
Sidebar (filtros)  |  Conteudo Principal
--------------------|--------------------
Canal              |  [KPI] [KPI] [KPI]
Categoria          |  [Grafico 1]       
Periodo            |  [Grafico 2]       
                   |  [Tabela]          
                   |  [Storytelling]    
```

In [ ]:
# ============================================================
# SECAO 8-9 - Gerar Codigo do App Streamlit
# ============================================================

app_code = '''# -*- coding: utf-8 -*-
"""Dashboard Analitico - Vendas do Varejo Brasileiro"""

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ============================================================
# CONFIGURACAO DA PAGINA
# ============================================================
st.set_page_config(
    page_title="Dashboard Varejo BR",
    page_icon="📊",
    layout="wide"
)

# ============================================================
# CARREGAMENTO DOS DADOS
# ============================================================
@st.cache_data
def carregar_dados():
    df = pd.read_csv("vendas_brasil_clean.csv")
    df["data"] = pd.to_datetime(df["data"])
    return df

df = carregar_dados()

# ============================================================
# SIDEBAR - FILTROS
# ============================================================
st.sidebar.header("Filtros")

# Filtro de Canal
canais = df["canal"].unique().tolist()
canal_selecionado = st.sidebar.multiselect("Canal de Venda", canais, default=canais)

# Filtro de Categoria
categorias = df["categoria"].unique().tolist()
categoria_selecionada = st.sidebar.multiselect("Categoria", categorias, default=categorias)

# Filtro de UF
ufs = df["uf"].unique().tolist()
uf_selecionada = st.sidebar.multiselect("Estado (UF)", ufs, default=ufs)

# Filtro de Periodo
data_min = df["data"].min().date()
data_max = df["data"].max().date()
periodo = st.sidebar.date_input("Periodo", value=(data_min, data_max),
                                 min_value=data_min, max_value=data_max)

# Aplicar filtros
df_filtro = df[
    (df["canal"].isin(canal_selecionado)) &
    (df["categoria"].isin(categoria_selecionada)) &
    (df["uf"].isin(uf_selecionada))
]

if len(periodo) == 2:
    df_filtro = df_filtro[
        (df_filtro["data"].dt.date >= periodo[0]) &
        (df_filtro["data"].dt.date <= periodo[1])
    ]

# ============================================================
# TITULO E KPIs
# ============================================================
st.title("Dashboard de Vendas - Varejo Brasileiro")
st.markdown("Analise completa do desempenho de vendas por canal, categoria e regiao.")

# KPIs no topo
col1, col2, col3, col4 = st.columns(4)

receita_total = df_filtro["receita"].sum()
lucro_total = df_filtro["lucro"].sum()
margem = (lucro_total / receita_total * 100) if receita_total > 0 else 0
ticket_medio = df_filtro["receita"].mean()

with col1:
    st.metric("Receita Total", f"R$ {receita_total:,.2f}")
with col2:
    st.metric("Lucro Total", f"R$ {lucro_total:,.2f}")
with col3:
    st.metric("Margem de Lucro", f"{margem:.1f}%")
with col4:
    st.metric("Ticket Medio", f"R$ {ticket_medio:,.2f}")

st.divider()

# ============================================================
# GRAFICOS
# ============================================================

# Linha 1: Evolucao temporal + Composicao por canal
col_a, col_b = st.columns(2)

with col_a:
    st.subheader("Evolucao Mensal da Receita")
    receita_mes = df_filtro.groupby(df_filtro["data"].dt.to_period("M"))["receita"].sum().reset_index()
    receita_mes["data_str"] = receita_mes["data"].astype(str)
    fig = px.line(receita_mes, x="data_str", y="receita", markers=True,
                  labels={"data_str": "Mes", "receita": "Receita (R$)"})
    fig.update_layout(xaxis_tickangle=-45)
    st.plotly_chart(fig, use_container_width=True)

with col_b:
    st.subheader("Composicao por Canal")
    canal_data = df_filtro.groupby("canal")["receita"].sum().reset_index()
    fig = px.pie(canal_data, names="canal", values="receita",
                 hole=0.4)
    st.plotly_chart(fig, use_container_width=True)

# Linha 2: Ranking por UF + Scatter
col_c, col_d = st.columns(2)

with col_c:
    st.subheader("Receita por Estado")
    uf_data = df_filtro.groupby("uf")["receita"].sum().sort_values(ascending=True).reset_index()
    fig = px.bar(uf_data, x="receita", y="uf", orientation="h",
                 labels={"uf": "Estado", "receita": "Receita (R$)"},
                 color="receita", color_continuous_scale="Blues")
    st.plotly_chart(fig, use_container_width=True)

with col_d:
    st.subheader("Receita vs Lucro")
    fig = px.scatter(df_filtro, x="receita", y="lucro", color="canal",
                     hover_data=["uf", "categoria"], opacity=0.6)
    fig.add_hline(y=0, line_dash="dash", line_color="red")
    st.plotly_chart(fig, use_container_width=True)

# ============================================================
# TABELA DE DADOS
# ============================================================
st.divider()
st.subheader("Dados Detalhados")
st.dataframe(df_filtro, use_container_width=True)

# Download
csv = df_filtro.to_csv(index=False).encode("utf-8")
st.download_button("下载 CSV", csv, "vendas_filtradas.csv", "text/csv")

# ============================================================
# STORYTELLING
# ============================================================
st.divider()
st.subheader("Insights Analiticos")

st.markdown(f"""
### Principais Descobertas

- **Receita Total:** R$ {receita_total:,.2f} no periodo filtrado
- **Margem de Lucro:** {margem:.1f}% - {'Saudavel' if margem > 10 else 'Atencao necessaria'}
- **Ticket Medio:** R$ {ticket_medio:,.2f} por transacao
- **Canal Lider:** {df_filtro.groupby("canal")["receita"].sum().idxmax()} com maior volume de vendas
- **Estado Lider:** {df_filtro.groupby("uf")["receita"].sum().idxmax()} em receita total

### Recomendacoes

1. Investir no canal com maior margem de lucro
2. Explorar oportunidades nos estados com menor participacao
3. Monitorar categorias com margem negativa
4. Avaliar sazonalidade para campanhas promocionais
""")
'''

with open('app_dashboard.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('app_dashboard.py criado com sucesso!')

---

## Secao 10: Git e Portfolio Profissional

*Corresponde a Aula 10: GitHub, Versionamento e Portfolio*

### Conceito-chave: Boas Praticas de Versionamento

```bash
# Inicializar repositorio
git init

# Adicionar arquivos
git add .

# Primeiro commit
git commit -m "feat: projeto integrador - analise de vendas"

# Criar repositorio no GitHub
gh repo create varejo-analise --public --source=. --push
```

### Estrutura Recomendada do Repositorio

```
varejo-analise/
|
|-- README.md
|-- requirements.txt
|-- vendas_brasil.csv
|-- vendas_brasil_clean.csv
|-- vendas.db
|-- projeto_integrador.ipynb
|-- app_dashboard.py
|-- .gitignore
```

---

## Secao 11: Deploy - Da Vitrine para o Mundo

*Corresponde a Aula 11: Deploy com Streamlit Cloud*

### Passo a Passo para Publicar

1. **Criar repositorio no GitHub** com todos os arquivos
2. **Criar arquivo requirements.txt**:
```txt
pandas
numpy
matplotlib
seaborn
plotly
sqlalchemy
streamlit
```
3. **Acessar** [share.streamlit.io](https://share.streamlit.io)
4. **Conectar** ao repositorio GitHub
5. **Selecionar** o arquivo `app_dashboard.py`
6. **Clicar** em Deploy!

### Resultado

Link definitivo tipo: `https://seu-app.streamlit.app/`

---

## Secao 12: O Pitch Analitico - Do Codigo a Decisao

*Corresponde a Aula 12: Do Codigo a Decisao*

### Estrutura do Pitch de Dados

#### 1. Contexto do Negocio
- Analise de vendas de varejo nacional
- 10 estados, 4 canais de venda, multiplas categorias
- Periodo: Janeiro 2023 a Dezembro 2024

#### 2. Qualidade dos Dados
- Pipeline completo: coleta -> limpeza -> analise
- Engenharia de recursos: margem, ticket, classificacao
- Modelagem relacional: tabelas Fato e Dimensao

#### 3. KPIs Apresentados

```python
# KPIs fundamentais calculados:
receita_total = df['receita'].sum()
lucro_total = df['lucro'].sum()
margem_lucro = (lucro_total / receita_total) * 100
ticket_medio = df['receita'].mean()
```

#### 4. Insights Acionaveis

- Identificar canais com maior margem de lucro
- Mapear estados com potencial de crescimento
- Monitorar categorias com desempenho inferior
- Planejar campanhas com base na sazonalidade

#### 5. Usabilidade ao Vivo

- Dashboard interativo com filtros dinamicos
- Graficos Plotly com hover e zoom
- Tabela de dados com download
- Publicacao via Streamlit Cloud

---

## Projeto Integrador - Conclusao

### O que foi feito neste projeto:

| Etapa | Descricao | Ferramenta |
|-------|-----------|------------|
| 1. Introducao | Pipeline de dados e setup | pandas |
| 2. Exploracao | Check-up e selecao de dados | pandas |
| 3. Limpeza | Tipos, nulos, feature engineering | pandas |
| 4. KPIs | Metricas de negocio | pandas |
| 5. Visualizacao Estatica | Graficos estatisticos | matplotlib, seaborn |
| 6. Visualizacao Interativa | Graficos dinamicos | plotly |
| 7. SQL + Python | Banco relacional | sqlalchemy |
| 8-9. Dashboard | Aplicacao web interativa | streamlit |
| 10. Git | Versionamento | git, GitHub |
| 11. Deploy | Publicacao na web | Streamlit Cloud |
| 12. Pitch | Apresentacao executiva | markdown |

### Arquivos Gerados

- `projeto_integrador.ipynb` - Notebook completo
- `vendas_brasil_clean.csv` - Dados tratados
- `vendas.db` - Banco SQLite
- `app_dashboard.py` - Dashboard Streamlit

### proximos Passos

1. Executar o notebook sequencialmente
2. Testar o app Streamlit localmente: `streamlit run app_dashboard.py`
3. Publicar no Streamlit Cloud
4. Compartilhar o link do portfolio